# End-to-end pipeline

This notebook runs the fact-checking pipeline from audio to stored results. Each stage reads from and writes to the SQLite database created by `database.ipynb`.

1. **Transcription:** transcribe the audio in overlapping windows, save the segments to `transcript_segments`, and regroup them into complete sentences for claim checking.

2. **Claim checking:** each complete sentence is checked with the AI service and the claims, verdicts, evidence and timings are stored. v0 runs this sequentially inside the transcription loop, as a naive baseline.

**Before running:**
- Run `database.ipynb` once to create `database/app.db`.
- Put the episode audio at `samples/planet_money_1000.mp3` (see `asr/podcast-offline-test.ipynb` for the source).

## Setup

In [1]:
import sqlite3
import time
from pathlib import Path

from faster_whisper import WhisperModel, decode_audio

c:\Users\alexi\HES-Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SAMPLES = Path("samples")
AUDIO_FILE = SAMPLES / "true_false.mp3"   # NPR Planet Money, Episode 1,000
DB_PATH = "database/app.db"                       # created by running database.ipynb

SAMPLE_RATE = 16000    # Whisper expects 16 kHz audio
WINDOW_S = 30          # length of each transcription window
OVERLAP_S = 5          # audio shared with the next window

## Database

`connect()` applies the same per-connection settings as `database.ipynb`. `check_database()` stops with a clear message if the database or its tables are missing.

In [3]:
REQUIRED_TABLES = {"sources", "transcript_segments", "claims", "claim_segments",
                   "verified_claims", "evidence", "verdict_evidence"}


def connect(db_path: str = DB_PATH) -> sqlite3.Connection:
    """Open the database with the per-connection settings from database.ipynb."""
    con = sqlite3.connect(db_path, timeout=5)
    con.execute("PRAGMA foreign_keys = ON")
    con.execute("PRAGMA synchronous = NORMAL")
    con.execute("PRAGMA busy_timeout = 5000")
    return con


def check_database(db_path: str = DB_PATH) -> None:
    """Stop with a clear message unless the database file exists and has the schema."""
    if not Path(db_path).exists():
        raise FileNotFoundError(f"{db_path} does not exist. Run database.ipynb first.")
    con = connect(db_path)
    try:
        tables = {r[0] for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table'")}
    finally:
        con.close()
    missing = REQUIRED_TABLES - tables
    if missing:
        raise RuntimeError(f"{db_path} is missing tables {sorted(missing)}. "
                           "Run database.ipynb first.")
    print(f"database OK: {db_path}")


check_database()

database OK: database/app.db


## 1. Transcription helpers

The audio is transcribed in overlapping windows. Each window owns the words that start in its first `WINDOW_S - OVERLAP_S` seconds; the overlap belongs to the next window, which hears it with more context. Whisper's word timestamps let the next window skip words the previous one already saved, so no words are repeated or lost at a boundary. Each window's segments are committed as soon as they are ready, so the database fills in while the loop runs.

In [4]:
model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=8)

In [5]:
audio = decode_audio(str(AUDIO_FILE), sampling_rate=SAMPLE_RATE)
print(f"{AUDIO_FILE.name}: {len(audio) / SAMPLE_RATE / 60:.1f} minutes")

planet_money_1000.mp3: 27.7 minutes


In [6]:
def transcribe_and_save(model, audio, source_url, on_sentence=None,
                        window_s=WINDOW_S, overlap_s=OVERLAP_S, db_path=DB_PATH):
    """Transcribe in overlapping windows, committing each window's segments as it finishes.

    Overlapping audio is resolved word by word: a window keeps the words that start in its
    share and skips words the previous window already saved. Committed segments are
    regrouped into sentences; each complete sentence is passed to
    on_sentence(source_id, sentence, available_at), where available_at (epoch ms) is when the
    sentence became complete: the start of its verification latency.
    Returns the new source id."""
    check_database(db_path)
    con = connect(db_path)
    assembler = SentenceAssembler()
    step_s = window_s - overlap_s
    total_s = len(audio) / SAMPLE_RATE
    source_id = None
    last_end_s = 0.0   # end of the last saved word, in episode time

    def emit(sentences, available_at):
        if on_sentence is not None:
            for sentence in sentences:
                on_sentence(source_id, sentence, available_at)

    try:
        # --------------------------------- Create new source in DB -----------------------------------------------
        with con:  # create the source row first, so the DB shows it straight away
            cur = con.execute(
                "INSERT INTO sources (source_url, status) VALUES (?, 'transcribing')",
                (source_url,),
            )
        source_id = cur.lastrowid

        # -------------------------------- Process each window ----------------------------------------------------
        start_s = 0.0
        while True:
            end_s = min(start_s + window_s, total_s)
            clip = audio[int(start_s * SAMPLE_RATE) : int(end_s * SAMPLE_RATE)]
            segments, _ = model.transcribe(clip, language="en", task="transcribe",
                                           beam_size=5, vad_filter=True, word_timestamps=True)
            is_last = end_s >= total_s
            owned_end = total_s if is_last else start_s + step_s   # this window's share

            saved = []   # (segment_id, start_ms, end_ms, text), in order
            with con:  # commit this window now, so readers see it while the loop runs
                for s in segments:
                    words = [
                        w for w in (s.words or [])
                        if start_s + w.start < owned_end                         # the next window owns the rest
                        and start_s + (w.start + w.end) / 2 >= last_end_s        # not already saved
                    ]
                    text = "".join(w.word for w in words).strip()
                    if not text:
                        continue
                    start_ms = round((start_s + words[0].start) * 1000)
                    end_ms = round((start_s + words[-1].end) * 1000)
                    cur = con.execute(
                        """INSERT INTO transcript_segments
                               (video_id, start_ms, end_ms, text, is_stable)
                           VALUES (?, ?, ?, ?, 1)""",
                        (source_id, start_ms, end_ms, text),
                    )
                    saved.append((cur.lastrowid, start_ms, end_ms, text))
                    last_end_s = max(last_end_s, start_s + words[-1].end)
            print(f"window {start_s:7.1f}-{end_s:7.1f} s: {len(saved)} segments saved")

            committed_at = time.time_ns() // 1_000_000   # this window's sentences are available now
            for segment in saved:   # after the commit, so the ids exist for claim_segments
                emit(assembler.add(*segment), committed_at)

            if is_last:
                break
            start_s += step_s

        emit(assembler.flush(), time.time_ns() // 1_000_000)   # the last sentence may have no closing punctuation

        with con:  # transcription done; the next stages pick it up from here
            con.execute("UPDATE sources SET status = 'processing' WHERE id = ?", (source_id,))
        return source_id
    except Exception:
        if source_id is not None:   # keep the windows already saved, mark the run as failed
            with con:
                con.execute("UPDATE sources SET status = 'failed' WHERE id = ?", (source_id,))
        raise
    finally:
        con.close()

### 1.1 Sentence assembly helpers

In [7]:
import warnings
from dataclasses import dataclass
from typing import cast

warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")   # harmless warnings from an old package
import pysbd
from pysbd.utils import TextSpan

_segmenter = pysbd.Segmenter(language="en", clean=False, char_span=True)


@dataclass
class Sentence:
    text: str
    start_ms: int
    end_ms: int
    segment_ids: list


def sentence_cuts(text):
    """Character positions where one sentence ends and the next begins.
    The last sentence is left out: it may still be unfinished."""
    spans = cast(list[TextSpan], _segmenter.segment(text))   # char_span=True returns TextSpans
    return [span.start for span in spans[1:]]


class SentenceAssembler:
    """Regroup transcript segments into complete sentences.

    Feed segments in order with add(); it returns the sentences they complete and keeps
    the unfinished tail until a later segment ends it. Call flush() at the end of the source."""

    def __init__(self, max_words=60):
        self.max_words = max_words   # cut anyway if no punctuation arrives for this long
        self.text = ""
        self.spans = []              # (start_char, end_char, segment_id, start_ms, end_ms)

    def add(self, segment_id, start_ms, end_ms, text):
        if self.text:
            self.text += " "
        start = len(self.text)
        self.text += text.strip()
        self.spans.append((start, len(self.text), segment_id, start_ms, end_ms))

        sentences, prev = [], 0
        for cut in sentence_cuts(self.text):
            sentences.append(self._make(prev, cut))
            prev = cut
        if len(self.text[prev:].split()) > self.max_words:
            sentences.append(self._make(prev, len(self.text)))
            prev = len(self.text)
        self._drop(prev)
        return sentences

    def flush(self):
        if not self.text.strip():
            return []
        sentences = [self._make(0, len(self.text))]
        self._drop(len(self.text))
        return sentences

    def _make(self, a, b):
        parts = [sp for sp in self.spans if sp[0] < b and sp[1] > a]
        return Sentence(self.text[a:b].strip(), parts[0][3], parts[-1][4], [p[2] for p in parts])

    def _drop(self, upto):
        """Forget the text before `upto`, keeping the unfinished tail."""
        self.text = self.text[upto:]
        self.spans = [(max(s - upto, 0), e - upto, sid, a, b)
                      for s, e, sid, a, b in self.spans if e > upto]

## 2. AI service helpers

Copied from `ai_agent.ipynb`, which the AI workstream owns: keep the two in sync until they move to a shared module. Two changes: the NLI model is named `nli_model` so it doesn't replace the Whisper `model`, and `nli()` takes a lock, so it stays safe when worker threads are added after v0.

In [ ]:
import os
import re
import threading

os.environ.setdefault("HF_HOME", os.path.abspath(".cache/huggingface"))   # same model cache as ai_agent.ipynb

import torch
from ddgs import DDGS
from trafilatura import extract, fetch_url
from transformers import AutoModelForSequenceClassification, AutoTokenizer

NLI_MODEL_NAME = "cross-encoder/nli-MiniLM2-L6-H768"
MAX_RESULTS = 3
MAX_PASSAGES = 5
EVIDENCE_STRENGTH_THRESHOLD = 0.60
CONFLICT_MARGIN = 0.10

# nli_* names, so they don't replace the Whisper `model` above
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
assert nli_tokenizer is not None   # from_pretrained only returns None on failure; this narrows the type
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()
_nli_lock = threading.Lock()   # fast tokenizers raise errors when two threads use them at once


def nli(premise, hypothesis):
    with _nli_lock:
        inputs = nli_tokenizer(
            premise, hypothesis, return_tensors="pt",
            truncation="only_first", max_length=512,
        )
        with torch.inference_mode():
            scores = nli_model(**inputs).logits.softmax(dim=-1)[0].tolist()
    return {nli_model.config.id2label[i].lower(): score for i, score in enumerate(scores)}


print(nli("Paris is in France.", "Paris is located in France."))

c:\Users\alexi\HES-Hackathon\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alexi\.cache\huggingface\hub\models--cross-encoder--nli-MiniLM2-L6-H768. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\alexi\HES-Hackathon\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarni

{'contradiction': 0.00046864381874911487, 'entailment': 0.9909460544586182, 'neutral': 0.008585324510931969}


In [9]:
QUESTION_STARTS = {
    "who", "what", "when", "where", "why", "how", "which",
    "is", "are", "do", "does", "did", "can", "could", "would", "should"
}
OPINION_MARKERS = {
    "i think", "i believe", "i feel", "in my opinion", "i prefer",
    "best", "worst", "beautiful", "ugly", "delicious", "boring", "amazing", "favorite"
}
COMMAND_STARTS = {
    "please", "tell", "give", "show", "explain", "close", "open",
    "stop", "go", "make", "write", "run", "click", "select"
}
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "has", "have", "in", "is", "it", "of", "on",
    "that", "the", "to", "was", "were", "with"
}


def is_checkable(text):
    clean = text.strip()
    words = re.findall(r"[a-z']+", clean.lower())
    first_word = words[0] if words else ""

    if len(words) < 3:
        return False
    if clean.endswith("?") or first_word in QUESTION_STARTS:
        return False
    if first_word in COMMAND_STARTS:
        return False
    if any(marker in clean.lower() for marker in OPINION_MARKERS):
        return False
    return True


def content_words(text):
    return set(re.findall(r"\w+", text.lower())) - STOPWORDS


def search_web(claim):
    return list(DDGS(timeout=10).text(claim, max_results=MAX_RESULTS))


def collect_evidence(claim, search_results):
    claim_words = content_words(claim)
    passages = []
    errors = []
    seen = set()

    def add_passage(text, url, source_type):
        passage = re.sub(r"\s+", " ", text).strip()[:700]
        key = passage.lower()
        if not passage or key in seen:
            return
        overlap = len(claim_words & content_words(passage))
        relevance = overlap / max(len(claim_words), 1)
        if relevance >= 0.5:
            seen.add(key)
            passages.append({
                "text": passage, "url": url, "source_type": source_type,
                "overlap": overlap, "relevance": round(relevance, 3),
            })

    for result in search_results:
        url = result.get("href") or result.get("url")
        if not url:
            continue
        snippet = result.get("body") or result.get("snippet") or ""
        if snippet:
            add_passage(snippet, url, "search_snippet")
        try:
            html = fetch_url(url)
            text = extract(html, include_comments=False) if html else None
        except Exception as error:
            errors.append(f"{url}: {error}")
            continue
        if not text:
            errors.append(f"{url}: no readable text")
            continue
        sentences = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text))
        for sentence in sentences:
            add_passage(sentence, url, "page")
    passages.sort(
        key=lambda passage: (passage["relevance"], passage["source_type"] == "page"),
        reverse=True,
    )
    return passages[:MAX_PASSAGES], errors


def check_claim(text):
    text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", text.strip(), flags=re.IGNORECASE)
    started = time.perf_counter()
    result = {"claim": text, "verdict": "not_checkable", "evidence": [],
              "errors": [], "timings_ms": {}}
    timings = result["timings_ms"]
    checkable = is_checkable(text)
    timings["checkability"] = (time.perf_counter() - started) * 1000

    if checkable:
        result["verdict"] = "uncertain"
        stage = time.perf_counter()
        try:
            hits = search_web(text)
        except Exception as error:
            hits = []
            result["errors"].append(f"Search failed: {error}")
        timings["search"] = (time.perf_counter() - stage) * 1000

        stage = time.perf_counter()
        passages, errors = collect_evidence(text, hits)
        result["errors"].extend(errors)
        timings["retrieval"] = (time.perf_counter() - stage) * 1000

        stage = time.perf_counter()
        for passage in passages:
            passage["scores"] = nli(passage["text"], text)
            passage["support_strength"] = round(
                passage["scores"]["entailment"] * passage["relevance"], 4
            )
            passage["contradiction_strength"] = round(
                passage["scores"]["contradiction"] * passage["relevance"], 4
            )
        if passages:
            support = max(passages, key=lambda p: p["support_strength"])
            against = max(passages, key=lambda p: p["contradiction_strength"])
            support_score = support["support_strength"]
            contradiction_score = against["contradiction_strength"]
            supported = support_score >= EVIDENCE_STRENGTH_THRESHOLD
            contradicted = contradiction_score >= EVIDENCE_STRENGTH_THRESHOLD
            close_conflict = abs(support_score - contradiction_score) < CONFLICT_MARGIN
            if supported and contradicted and close_conflict:
                result["evidence"] = [support, against]
            elif supported and support_score > contradiction_score:
                result.update(verdict="supported", evidence=[support])
            elif contradicted:
                result.update(verdict="contradicted", evidence=[against])
            else:
                result["evidence"] = passages
        timings["verification"] = (time.perf_counter() - stage) * 1000

    timings["total"] = (time.perf_counter() - started) * 1000
    result["timings_ms"] = {name: round(ms, 2) for name, ms in timings.items()}
    return result

### 2.1 Claim storage helpers

`insert_checking_claim()` stores a checkable sentence as a `checking` claim linked to its transcript segments. `save_result()` stores the verdict, evidence, NLI scores and stage timings, and sets `verdict_at`: verification latency is `verdict_at - stable_at`.

In [10]:
import json

MODEL_VERSION = f"{NLI_MODEL_NAME}+rules-v0"   # stored with each verdict, to tell cached verdicts apart later
VERDICT_TTL_MS = 7 * 24 * 60 * 60 * 1000       # how long a verdict stays reusable in verified_claims


def now_ms():
    """The shared clock from database.ipynb: epoch milliseconds."""
    return time.time_ns() // 1_000_000


def passage_stance(passage):
    """Stance of one evidence passage, using the same threshold as check_claim."""
    support, against = passage["support_strength"], passage["contradiction_strength"]
    if support >= EVIDENCE_STRENGTH_THRESHOLD and support > against:
        return "supports"
    if against >= EVIDENCE_STRENGTH_THRESHOLD:
        return "contradicts"
    return "neutral"


def insert_checking_claim(con, source_id, sentence, stable_at):
    """Store a sentence as a claim being checked, linked to its transcript segments."""
    with con:
        cur = con.execute(
            """INSERT INTO claims (video_id, claim_text, start_ms, end_ms, status, stable_at)
               VALUES (?, ?, ?, ?, 'checking', ?)""",
            (source_id, sentence.text, sentence.start_ms, sentence.end_ms, stable_at),
        )
        claim_id = cur.lastrowid
        con.executemany(
            "INSERT INTO claim_segments (claim_id, segment_id) VALUES (?, ?)",
            [(claim_id, segment_id) for segment_id in dict.fromkeys(sentence.segment_ids)],
        )
    return claim_id


def save_result(con, claim_id, result):
    """Store check_claim's verdict, evidence and timings; this stops the latency stopwatch."""
    verdict_at = now_ms()
    timings_json = json.dumps(result["timings_ms"])
    with con:
        if result["verdict"] == "not_checkable":
            con.execute(
                "UPDATE claims SET status = 'not_checked', verdict_at = ?, timings_json = ? WHERE id = ?",
                (verdict_at, timings_json, claim_id),
            )
            return

        cur = con.execute(
            """INSERT INTO verified_claims (claim_text, verdict, verified_at, expires_at, model_version)
               VALUES (?, ?, ?, ?, ?)""",
            (result["claim"], result["verdict"], verdict_at, verdict_at + VERDICT_TTL_MS, MODEL_VERSION),
        )
        verified_claim_id = cur.lastrowid

        for passage in result["evidence"]:
            cur = con.execute(
                "INSERT INTO evidence (evidence_text, source_url, fetched_at) VALUES (?, ?, ?)",
                (passage["text"], passage["url"], verdict_at),   # fetched during this check
            )
            scores = {
                "relevance": passage["relevance"],
                **passage["scores"],                              # entailment, neutral, contradiction
                "support_strength": passage["support_strength"],
                "contradiction_strength": passage["contradiction_strength"],
            }
            con.execute(
                """INSERT INTO verdict_evidence (verified_claim_id, evidence_id, stance, scores_json)
                   VALUES (?, ?, ?, ?)""",
                (verified_claim_id, cur.lastrowid, passage_stance(passage), json.dumps(scores)),
            )

        con.execute(
            """UPDATE claims
               SET status = ?, verified_claim_id = ?, path = 'fresh', verdict_at = ?, timings_json = ?
               WHERE id = ?""",
            (result["verdict"], verified_claim_id, verdict_at, timings_json, claim_id),
        )

### 2.2 Sequential checking (v0)

v0 has no concurrency: the transcription loop checks each complete sentence itself before moving on to the next window. `stable_at` is when the sentence became complete (its window was committed), so the measured latency includes the time a claim waits behind earlier claims. Repeats of recent sentences are skipped.

In [11]:
from collections import deque


def check_sentence(con, source_id, sentence, stable_at):
    """Check one sentence and store the result; sentences that aren't claims are skipped."""
    claim_id = None
    try:
        if not is_checkable(sentence.text):
            return   # not a claim: nothing stored, the passage stays "Not checked"
        claim_id = insert_checking_claim(con, source_id, sentence, stable_at)
        result = check_claim(sentence.text)
        save_result(con, claim_id, result)
        print(f"claim {claim_id}: {result['verdict']:<13}"
              f"{result['timings_ms']['total']:7.0f} ms  {sentence.text}")
    except Exception as error:   # one failed claim shouldn't stop the run
        print(f"claim {claim_id} failed: {error!r}  {sentence.text}")
        if claim_id is not None:
            with con:
                con.execute("UPDATE claims SET status = 'not_checked' WHERE id = ?", (claim_id,))


def make_checker(con, recent=5):
    """on_sentence callback for v0: check each sentence right away, inside the transcription
    loop, skipping repeats of recent sentences."""
    last = deque(maxlen=recent)

    def on_sentence(source_id, sentence, available_at):
        key = sentence.text.lower()
        if key in last:
            return   # e.g. six "The Great Depression." in a row
        last.append(key)
        check_sentence(con, source_id, sentence, available_at)
    return on_sentence

## Operational loop

In [12]:
import math
import statistics
from collections import Counter

t0 = time.perf_counter()
con = connect()   # for the claim writes; transcribe_and_save opens its own for the segments
try:
    source_id = transcribe_and_save(model, audio, AUDIO_FILE.name, on_sentence=make_checker(con))
    with con:
        con.execute("UPDATE sources SET status = 'done' WHERE id = ? AND status = 'processing'",
                    (source_id,))
    rows = con.execute(
        "SELECT status, verdict_at - stable_at FROM claims WHERE video_id = ? AND verdict_at IS NOT NULL",
        (source_id,),
    ).fetchall()
finally:
    con.close()
elapsed = time.perf_counter() - t0

latencies = sorted(ms for _, ms in rows)
print(f"\nsource {source_id}: {len(rows)} claims checked {dict(Counter(status for status, _ in rows))}")
if latencies:
    p95 = latencies[math.ceil(0.95 * len(latencies)) - 1]
    print(f"verification latency  p50 {statistics.median(latencies):.0f} ms  p95 {p95:.0f} ms  "
          f"under 1 s {100 * sum(ms < 1000 for ms in latencies) / len(latencies):.0f} %")
audio_s = len(audio) / SAMPLE_RATE
print(f"total {elapsed:.0f} s for {audio_s:.0f} s of audio  (RTF {elapsed / audio_s:.2f}, transcription + checking)")

database OK: database/app.db
window     0.0-   30.0 s: 5 segments saved
claim 1: uncertain       7734 ms  This week on Up First, the midterms less than 15 days away.
claim 2: uncertain       4069 ms  President Trump, Vice President J.D. Vance, and former Presi
window    25.0-   55.0 s: 5 segments saved
claim 3: uncertain      12219 ms  Join us every morning as we guide you through the home stret
claim 4: uncertain      19960 ms  This is Planet Money from NPR.
claim 5: supported      13421 ms  Yeah, it's our 1,000th episode.
claim 6: uncertain      16272 ms  And to be honest, we spent a lot of time thinking about ways
window    50.0-   80.0 s: 12 segments saved
claim 7: supported      73376 ms  There's a lot going on right now.
claim 8: uncertain       7966 ms  Or my personal suggestion, we could suddenly switch to decim
claim 9: contradicted    9942 ms  We don't ever have to get to 1,000.
claim 10: supported      13484 ms  We can stay young forever.
window    75.0-  105.0 s: 8 segments

KeyboardInterrupt: 